# JetBot tự tránh vị trí nguy hiểm

## Nhiệm vụ
JetBot quan sát ảnh camera phía trước, phân loại cảnh hiện tại là **an toàn** hoặc **nguy hiểm**, rồi tự dừng trước khi đi vào vị trí nguy hiểm. Đây là bài học thị giác máy tính theo quy trình: **xác định nhiệm vụ → thu thập ảnh → gắn nhãn → kiểm tra dữ liệu → huấn luyện → đánh giá → chạy thực tế → cải tiến**.

Notebook tham khảo cách học theo thử thách, thực hành và kiểm tra dần của [CS2N – Carnegie Mellon Robotics Academy](https://www.cs2n.org/). Mô hình và cấu trúc thư mục tương thích với các notebook `collision_avoidance` có sẵn trong dự án này.

**Cách làm:** chạy các ô theo thứ tự từ trên xuống. Chỉ chuyển bước khi ô hiện tại in thông báo thành công hoặc hướng dẫn rõ bước tiếp theo. Không chạy phần 8 nếu chưa hoàn thành phần 6 và phần 7.

## 0. An toàn trước khi chạy

- Đặt sa hình trên sàn phẳng, không gần cầu thang, mép bàn, người hoặc vật dễ vỡ.
- Lần đầu kiểm tra động cơ phải kê bánh xe lên giá đỡ.
- Một người luôn đứng gần để bấm **Dừng khẩn cấp** hoặc tắt nguồn nếu cần.
- Bài này chỉ dùng camera nên không đo được khoảng cách thật. JetBot có thể nhận sai; luôn chạy tốc độ thấp và không dùng robot gần vùng nguy hiểm thật.
- Nhãn `nguy hiểm` nghĩa là: nếu JetBot tiếp tục đi thẳng ở tốc độ đã chọn thì có khả năng va chạm, rơi khỏi mép hoặc đi vào vùng cấm.

## 1. Chuẩn bị sa hình và kế hoạch dữ liệu

Tạo ít nhất hai kiểu cảnh. **An toàn**: đường phía trước còn trống và JetBot có thể đi tiếp. **Nguy hiểm**: vật cản gần, vạch đỏ/vùng cấm, mép sa hình hoặc góc tường mà JetBot không được đi tới.

Thu tối thiểu 150 ảnh mỗi nhãn, ưu tiên 250–400 ảnh mỗi nhãn. Mỗi nhãn cần có nhiều góc nhìn, khoảng cách, ánh sáng, bóng đổ và nền khác nhau. Không chỉ chụp liên tiếp một vị trí vì ảnh gần giống nhau làm điểm kiểm tra cao giả tạo.

## 2. Khởi tạo camera và thư mục dữ liệu
Chạy ô này một lần. Ảnh sẽ được lưu cục bộ trên JetBot tại `dataset_safety/an_toan` và `dataset_safety/nguy_hiem`.

In [ ]:
import os
import time
from uuid import uuid4

import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, Heartbeat, bgr8_to_jpeg

DATASET_DIR = 'dataset_safety'
SAFE_DIR = os.path.join(DATASET_DIR, 'an_toan')
DANGER_DIR = os.path.join(DATASET_DIR, 'nguy_hiem')
MODEL_PATH = 'best_safety_model.pth'

os.makedirs(SAFE_DIR, exist_ok=True)
os.makedirs(DANGER_DIR, exist_ok=True)

camera = Camera.instance(width=224, height=224)
robot = Robot()
robot.stop()

camera_image = widgets.Image(format='jpeg', width=224, height=224)
camera_link = traitlets.dlink((camera, 'value'), (camera_image, 'value'), transform=bgr8_to_jpeg)
display(camera_image)
print('Camera đã sẵn sàng. JetBot đang dừng.')
print('Nếu chưa thấy ảnh sau 5 giây: kiểm tra camera, rồi chọn Kernel > Restart Kernel và chạy lại từ ô đầu tiên.')


## 3. Thu thập và gắn nhãn ảnh
Đặt JetBot tại một vị trí trong sa hình, xem ảnh camera rồi bấm đúng nút. Mỗi lần bấm lưu một ảnh. Thay đổi vị trí JetBot hoặc vật cản trước mỗi vài ảnh. Khi nhãn không chắc chắn, **không lưu ảnh đó**.

In [ ]:
button_layout = widgets.Layout(width='190px', height='52px')
safe_button = widgets.Button(description='Lưu ảnh AN TOÀN', button_style='success', layout=button_layout)
danger_button = widgets.Button(description='Lưu ảnh NGUY HIỂM', button_style='danger', layout=button_layout)
safe_count = widgets.IntText(value=len(os.listdir(SAFE_DIR)), description='An toàn:', disabled=True)
danger_count = widgets.IntText(value=len(os.listdir(DANGER_DIR)), description='Nguy hiểm:', disabled=True)
collection_status = widgets.HTML(value='<b>Hãy quan sát ảnh trước khi lưu.</b>')

def save_snapshot(directory):
    if not camera_image.value:
        collection_status.value = '<b style="color:red">Chưa có ảnh camera. Chờ vài giây rồi thử lại.</b>'
        return False
    image_path = os.path.join(directory, str(uuid4()) + '.jpg')
    with open(image_path, 'wb') as image_file:
        image_file.write(camera_image.value)
    return True

def save_safe(button):
    if save_snapshot(SAFE_DIR):
        safe_count.value = len(os.listdir(SAFE_DIR))
        collection_status.value = '<b style="color:green">Đã lưu ảnh an toàn.</b>'

def save_danger(button):
    if save_snapshot(DANGER_DIR):
        danger_count.value = len(os.listdir(DANGER_DIR))
        collection_status.value = '<b style="color:red">Đã lưu ảnh nguy hiểm.</b>'

safe_button.on_click(save_safe)
danger_button.on_click(save_danger)
display(widgets.HBox([safe_button, danger_button]))
display(widgets.HBox([safe_count, danger_count]))
display(collection_status)


## 4. Kiểm tra dữ liệu trước khi huấn luyện
Chạy ô này sau khi thu ảnh. Hai lớp nên có số lượng gần nhau; nếu một lớp ít hơn nhiều, tiếp tục thu lớp đó. Mở vài ảnh ngẫu nhiên để phát hiện ảnh đen, mờ hoặc gắn nhãn nhầm. **Nếu kết quả báo chưa đủ dữ liệu hoặc lệch nhãn, quay lại phần 3; chưa chạy phần 5.**

In [ ]:
import random
from IPython.display import Image

safe_files = [name for name in os.listdir(SAFE_DIR) if name.lower().endswith(('.jpg', '.jpeg', '.png'))]
danger_files = [name for name in os.listdir(DANGER_DIR) if name.lower().endswith(('.jpg', '.jpeg', '.png'))]
print('Số ảnh an toàn:', len(safe_files))
print('Số ảnh nguy hiểm:', len(danger_files))

if min(len(safe_files), len(danger_files)) < 50:
    print('Cần ít nhất 50 ảnh ở MỖI lớp để thử huấn luyện; khuyến nghị từ 150 ảnh mỗi lớp.')
elif max(len(safe_files), len(danger_files)) > 2 * min(len(safe_files), len(danger_files)):
    print('Cảnh báo: hai lớp lệch nhiều. Hãy thu thêm ảnh cho lớp ít hơn.')
else:
    print('Số lượng hai lớp tương đối cân bằng. Có thể chuyển sang huấn luyện.')

for label, directory, files in [('AN TOÀN', SAFE_DIR, safe_files), ('NGUY HIỂM', DANGER_DIR, danger_files)]:
    if files:
        sample_path = os.path.join(directory, random.choice(files))
        print(label, '-', sample_path)
        display(Image(filename=sample_path, width=224))


## 5. Huấn luyện mô hình
Mô hình học từ ảnh đã gắn nhãn. Ta dùng AlexNet đã được huấn luyện trước trên ảnh phổ biến, thay lớp cuối để nhận hai lớp của sa hình. Tập kiểm tra được tách riêng để đo khả năng dự đoán trên ảnh mô hình chưa thấy. Ảnh kiểm tra không được làm biến đổi màu để điểm đánh giá phản ánh đúng dữ liệu gốc.

Lưu ý: lần đầu chạy `pretrained=True`, JetBot có thể cần Internet để tải trọng số ban đầu. Nếu không tải được, kết nối JetBot Internet rồi chạy lại ô này.

In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms

if not torch.cuda.is_available():
    raise RuntimeError('Không tìm thấy CUDA. Notebook này cần chạy trên JetBot/Jetson có GPU.')

device = torch.device('cuda')
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_base = datasets.ImageFolder(DATASET_DIR, transform=train_transform)
test_base = datasets.ImageFolder(DATASET_DIR, transform=eval_transform)
print('Các lớp theo thứ tự mô hình:', train_base.classes)
if train_base.classes != ['an_toan', 'nguy_hiem']:
    raise RuntimeError('Cần đúng hai thư mục dataset_safety/an_toan và dataset_safety/nguy_hiem.')
class_counts = [train_base.targets.count(0), train_base.targets.count(1)]
print('Số ảnh theo lớp [an_toan, nguy_hiem]:', class_counts)
if min(class_counts) < 50:
    raise RuntimeError('Dữ liệu quá ít ở một nhãn. Thu ít nhất 50 ảnh cho MỖI nhãn trước khi huấn luyện.')
if max(class_counts) > 2 * min(class_counts):
    raise RuntimeError('Hai nhãn lệch quá nhiều. Hãy thu thêm ảnh cho nhãn ít hơn trước khi huấn luyện.')

# Chia riêng từng nhãn để cả tập huấn luyện và tập kiểm tra đều có ảnh an toàn/nguy hiểm.
import random
rng = random.Random(42)
train_indices, test_indices = [], []
for class_index in [0, 1]:
    class_indices = [index for index, target in enumerate(train_base.targets) if target == class_index]
    rng.shuffle(class_indices)
    class_test_size = max(10, int(0.2 * len(class_indices)))
    test_indices.extend(class_indices[:class_test_size])
    train_indices.extend(class_indices[class_test_size:])

train_dataset = torch.utils.data.Subset(train_base, train_indices)
test_dataset = torch.utils.data.Subset(test_base, test_indices)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)
print('Ảnh huấn luyện:', len(train_dataset), '| Ảnh kiểm tra:', len(test_dataset))


In [ ]:
# Dùng cách gọi tương thích với torchvision đi kèm JetBot.
model = models.alexnet(pretrained=True)
model.classifier[6] = torch.nn.Linear(model.classifier[6].in_features, 2)
model = model.to(device)

NUM_EPOCHS = 20
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))
            correct += (outputs.argmax(1).cpu() == labels).sum().item()

    accuracy = correct / float(len(test_dataset))
    average_loss = train_loss / float(len(train_dataset))
    print('Epoch {}/{} | loss: {:.4f} | kiểm tra: {:.1%}'.format(epoch + 1, NUM_EPOCHS, average_loss, accuracy))
    if accuracy >= best_accuracy:
        torch.save(model.state_dict(), MODEL_PATH)
        best_accuracy = accuracy

print('Đã lưu mô hình tốt nhất:', MODEL_PATH, '| Độ chính xác kiểm tra tốt nhất: {:.1%}'.format(best_accuracy))


## 6. Đánh giá trước khi chạy robot
Độ chính xác cao không đủ để khẳng định an toàn. Hãy đưa camera vào những tình huống mới: ánh sáng khác, vật cản khác, vị trí sát vùng cấm. Mục tiêu quan trọng nhất là **không bỏ sót cảnh nguy hiểm**. Nếu mô hình đoán nguy hiểm là an toàn, thu thêm ảnh của tình huống đó, gắn nhãn đúng và huấn luyện lại. **Không chạy phần 8 khi tỷ lệ phát hiện nguy hiểm dưới 90%.**

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

def evaluate_loader(loader):
    # Hàng là nhãn thật, cột là nhãn dự đoán: an_toan, nguy_hiem.
    matrix = torch.zeros(2, 2, dtype=torch.int64)
    with torch.no_grad():
        for images, labels in loader:
            predictions = model(images.to(device)).argmax(1).cpu()
            for actual, predicted in zip(labels, predictions):
                matrix[actual, predicted] += 1
    return matrix

confusion = evaluate_loader(test_loader)
print('                 Dự đoán an toàn | Dự đoán nguy hiểm')
print('Thật an toàn:    {:>16} | {:>19}'.format(confusion[0, 0].item(), confusion[0, 1].item()))
print('Thật nguy hiểm:  {:>16} | {:>19}'.format(confusion[1, 0].item(), confusion[1, 1].item()))
danger_total = confusion[1].sum().item()
danger_recall = confusion[1, 1].item() / float(danger_total) if danger_total else 0.0
print('Tỷ lệ phát hiện nguy hiểm: {:.1%}'.format(danger_recall))
if danger_recall < 0.90:
    print('CHƯA NÊN CHẠY TỰ ĐỘNG: hãy thu thêm ảnh nguy hiểm đa dạng rồi huấn luyện lại.')
else:
    print('Kết quả ban đầu đạt yêu cầu. Vẫn phải chạy thử ở tốc độ thấp, có người giám sát.')


## 7. Chạy thử thực tế có giám sát
Ô sau chỉ nạp mô hình và hiển thị xác suất nguy hiểm; robot **chưa di chuyển**. Đưa lần lượt cảnh an toàn và nguy hiểm vào trước camera để xem mô hình phản hồi.

In [ ]:
import cv2
import numpy as np

mean = 255.0 * np.array([0.485, 0.456, 0.406])
stdev = 255.0 * np.array([0.229, 0.224, 0.225])
normalize = torchvision.transforms.Normalize(mean, stdev)

def preprocess(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(rgb.transpose((2, 0, 1))).float()
    tensor = normalize(tensor).to(device)
    return tensor[None, ...]

def danger_probability(frame):
    with torch.no_grad():
        probabilities = F.softmax(model(preprocess(frame)), dim=1)
    return float(probabilities[0, 1])  # Lớp 1 là nguy_hiem.

probability_bar = widgets.FloatProgress(value=0, min=0, max=1, description='Nguy hiểm:', bar_style='')
probability_text = widgets.HTML(value='Đang chờ ảnh camera...')
display(widgets.VBox([probability_bar, probability_text]))

def preview_prediction(change):
    probability = danger_probability(change['new'])
    probability_bar.value = probability
    probability_bar.bar_style = 'danger' if probability >= 0.5 else 'success'
    probability_text.value = '<b>Xác suất nguy hiểm: {:.0%}</b>'.format(probability)

camera.observe(preview_prediction, names='value')
preview_prediction({'new': camera.value})
print('Đang ở chế độ xem trước: robot không tự chạy.')


## 8. Bật chế độ tự chạy chậm
Chỉ chạy ô này sau khi đã kiểm tra phần 7 và đạt yêu cầu phần 6. JetBot đi thẳng rất chậm khi xác suất nguy hiểm thấp; khi nguy hiểm vượt ngưỡng ở nhiều khung hình liên tiếp, bot dừng và khóa lại. Muốn chạy lại, đặt sa hình an toàn rồi bấm **Bắt đầu chạy**.

Không tăng `RUN_SPEED` quá `0.20` trong bài học này. Nếu bot dừng muộn, giảm tốc độ hoặc tăng số ảnh nguy hiểm gần vật cản/mép sa hình.

In [ ]:
# Bỏ chế độ xem trước để tránh hai hàm cùng xử lý một khung hình.
try:
    camera.unobserve(preview_prediction, names='value')
except ValueError:
    pass

if 'danger_recall' not in globals() or danger_recall < 0.90:
    raise RuntimeError('Chưa được bật tự chạy: cần hoàn thành phần 6 với tỷ lệ phát hiện nguy hiểm từ 90% trở lên.')

RUN_SPEED = 0.12
DANGER_THRESHOLD = 0.55
CONSECUTIVE_DANGER_FRAMES = 3
autonomous_running = False
danger_frames = 0

start_button = widgets.Button(description='Bắt đầu chạy', button_style='success', layout=widgets.Layout(width='160px'))
stop_button = widgets.Button(description='Dừng khẩn cấp', button_style='danger', layout=widgets.Layout(width='160px'))
run_status = widgets.HTML(value='<b>Robot đang dừng. Kiểm tra sa hình trước khi bấm Bắt đầu chạy.</b>')
display(widgets.HBox([start_button, stop_button]))
display(run_status)

def stop_autonomous(message):
    global autonomous_running, danger_frames
    autonomous_running = False
    danger_frames = 0
    robot.stop()
    run_status.value = '<b style="color:red">{}</b>'.format(message)

def start_autonomous(button):
    global autonomous_running, danger_frames
    robot.stop()
    autonomous_running = True
    danger_frames = 0
    run_status.value = '<b style="color:green">Đang chạy chậm, có giám sát.</b>'

def emergency_stop(button):
    stop_autonomous('ĐÃ DỪNG KHẨN CẤP')

def autonomous_update(change):
    global danger_frames
    probability = danger_probability(change['new'])
    probability_bar.value = probability
    probability_bar.bar_style = 'danger' if probability >= DANGER_THRESHOLD else 'success'

    if not autonomous_running:
        return
    if probability >= DANGER_THRESHOLD:
        danger_frames += 1
    else:
        danger_frames = 0

    if danger_frames >= CONSECUTIVE_DANGER_FRAMES:
        stop_autonomous('Phát hiện vị trí nguy hiểm ({:.0%}). JetBot đã dừng.'.format(probability))
    else:
        robot.forward(RUN_SPEED)
        run_status.value = 'Đang chạy | Xác suất nguy hiểm: {:.0%} | Khung nguy hiểm liên tiếp: {}/{}'.format(probability, danger_frames, CONSECUTIVE_DANGER_FRAMES)

start_button.on_click(start_autonomous)
stop_button.on_click(emergency_stop)
camera.observe(autonomous_update, names='value')
print('Đã gắn chế độ tự chạy. Robot vẫn đứng yên cho đến khi bấm Bắt đầu chạy.')


## 9. Dừng an toàn và cải tiến
Khi kết thúc, chạy ô dưới để dừng robot, ngắt bộ xử lý camera và giải phóng camera cho notebook khác.

Để cải tiến mô hình: ghi lại tình huống bot nhận sai, thu ảnh mới cho đúng nhãn, kiểm tra lại sự cân bằng hai lớp, rồi huấn luyện lại. Một cách nâng cao là thêm cảm biến khoảng cách (ultrasonic/ToF) làm lớp bảo vệ thứ hai thay vì chỉ dựa vào ảnh camera.

In [ ]:
robot.stop()
for callback in [preview_prediction if 'preview_prediction' in globals() else None, autonomous_update if 'autonomous_update' in globals() else None]:
    if callback is not None:
        try:
            camera.unobserve(callback, names='value')
        except ValueError:
            pass

camera_link.unlink()
camera.stop()
print('JetBot đã dừng và camera đã được giải phóng.')
